# Notebook 01 — Data Preparation

## Objective

Create a canonical, auditable dataset from raw stock CSV files without
constructing labels, splitting Train/Test, selecting the security universe,
imputing missing values, or making model-selection decisions.

## Local execution design

This notebook is designed for VS Code Jupyter and does not require Conda.

The repository root is also the default data root. Therefore, with the project at:

```text
E:/Iran_stock_trade/financial-ml-decision-support
```

raw data are read automatically from:

```text
E:/Iran_stock_trade/financial-ml-decision-support/raw_data
```

## This notebook performs

1. repository and data-path resolution;
2. configuration loading;
3. raw-file inventory and optional SHA-256 hashing;
4. required-column validation;
5. date parsing and chronological sorting;
6. deterministic duplicate-date removal;
7. numeric coercion;
8. transparent row-quality checks;
9. separation of metadata, candidate model inputs, and legacy audit fields;
10. canonical file generation;
11. audit-table and reproducibility-manifest generation.

## Explicitly deferred

- Temporal Train/Test split: Notebook 02
- New label reconstruction: Notebook 03
- Leakage and feature approval: Notebook 04
- Validation folds: Notebook 05
- Model selection: Notebook 06

> The legacy `class` column is not used as the new modeling target.

In [1]:

from __future__ import annotations

from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


Python: 3.12.2
pandas: 2.2.3
numpy: 1.26.4


## 1. Locate the repository and import project modules

In [2]:

def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Run this notebook from inside the project."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.data.loading import normalize_symbol, parse_market_dates, read_raw_stock_csv
from src.data.manifest import build_file_inventory
from src.data.validation import (
    build_row_quality_flags,
    coerce_numeric_columns,
    missing_required_columns,
    summarize_row_quality,
)
from src.utils.paths import data_paths, repository_result_paths, resolve_data_root
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


## 2. Load configuration

In [3]:

def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
columns_config = load_yaml(REPOSITORY_ROOT / "configs" / "columns.yaml")

DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(REPOSITORY_ROOT, paths_config)

RAW_DATA_PATH = DATA_PATHS["raw_data"]
PREPARED_DATA_PATH = DATA_PATHS["prepared_data"]
LEGACY_AUDIT_PATH = DATA_PATHS["legacy_audit_data"]

METADATA_COLUMNS = list(columns_config["metadata"])
CANDIDATE_FEATURES = list(columns_config["candidate_features"])
LEGACY_AUDIT_OPTIONAL = list(columns_config["legacy_audit_optional"])
PRICE_COLUMNS = dict(columns_config["price_columns"])

REQUIRED_COLUMNS = METADATA_COLUMNS + CANDIDATE_FEATURES
NUMERIC_COLUMNS = CANDIDATE_FEATURES + LEGACY_AUDIT_OPTIONAL

CALCULATE_INPUT_HASHES = False
OUTPUT_ENCODING = "utf-8-sig"

print("Data root:", DATA_ROOT)
print("Raw data:", RAW_DATA_PATH)
print("Prepared data:", PREPARED_DATA_PATH)
print("Legacy audit data:", LEGACY_AUDIT_PATH)
print("Candidate features:", len(CANDIDATE_FEATURES))


Data root: E:\Iran_stock_trade\financial-ml-decision-support
Raw data: E:\Iran_stock_trade\financial-ml-decision-support\raw_data
Prepared data: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\prepared
Legacy audit data: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\legacy_audit
Candidate features: 31


## 3. Inventory the immutable raw-data snapshot

In [4]:

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw-data directory does not exist: {RAW_DATA_PATH}\n"
        "Set FINML_DATA_ROOT or edit configs/paths.yaml."
    )

raw_files = sorted(RAW_DATA_PATH.glob("*.csv"))

if not raw_files:
    raise FileNotFoundError(f"No CSV files were found in: {RAW_DATA_PATH}")

file_inventory_df = build_file_inventory(
    raw_files,
    calculate_hashes=CALCULATE_INPUT_HASHES,
)

file_inventory_path = RESULT_PATHS["manifests"] / "01_input_file_inventory.csv"
file_inventory_df.to_csv(
    file_inventory_path,
    index=False,
    encoding=OUTPUT_ENCODING,
)

input_manifest_hash = stable_object_hash(
    file_inventory_df.fillna("").to_dict(orient="records")
)

print("Raw CSV files:", len(raw_files))
print("Input inventory:", file_inventory_path)
print("Input manifest hash:", input_manifest_hash)


Raw CSV files: 825
Input inventory: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\01_input_file_inventory.csv
Input manifest hash: 1325e202a3686becd739694c4c49902955978b33aeff9357fd436894486c7197


## 4. Prepare one file

In [5]:

def prepare_one_file(file_path: Path):
    raw_df = read_raw_stock_csv(file_path)

    schema_audit = {
        "file_name": file_path.name,
        "raw_rows": len(raw_df),
        "raw_columns": len(raw_df.columns),
    }

    missing_columns = missing_required_columns(raw_df, REQUIRED_COLUMNS)
    schema_audit["missing_required_columns"] = "|".join(missing_columns)
    schema_audit["required_schema_valid"] = not missing_columns

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    working_df = raw_df.copy()

    working_df["dEven"] = parse_market_dates(working_df["dEven"])
    invalid_date_rows = int(working_df["dEven"].isna().sum())
    working_df = working_df.dropna(subset=["dEven"]).copy()

    working_df = coerce_numeric_columns(
        working_df,
        [column for column in NUMERIC_COLUMNS if column in working_df.columns],
    )

    working_df = working_df.sort_values("dEven").reset_index(drop=True)
    duplicate_date_rows = int(
        working_df.duplicated(subset=["dEven"], keep="last").sum()
    )
    working_df = (
        working_df
        .drop_duplicates(subset=["dEven"], keep="last")
        .sort_values("dEven")
        .reset_index(drop=True)
    )

    working_df = build_row_quality_flags(
        working_df,
        price_columns=PRICE_COLUMNS,
        candidate_features=CANDIDATE_FEATURES,
    )

    symbol = normalize_symbol(file_path.stem)

    chronology_audit = {
        "symbol": symbol,
        "file_name": file_path.name,
        "invalid_date_rows_removed": invalid_date_rows,
        "duplicate_date_rows_removed": duplicate_date_rows,
        "prepared_rows": len(working_df),
        "first_date": working_df["dEven"].min(),
        "last_date": working_df["dEven"].max(),
        "chronological_order_valid": bool(working_df["dEven"].is_monotonic_increasing),
        "unique_dates_valid": bool(not working_df["dEven"].duplicated().any()),
    }

    quality_audit = {
        "symbol": symbol,
        **summarize_row_quality(
            working_df,
            candidate_feature_count=len(CANDIDATE_FEATURES),
        ),
    }

    quality_columns = [
        column for column in working_df.columns if column.startswith("quality_")
    ]
    prepared_columns = METADATA_COLUMNS + CANDIDATE_FEATURES + quality_columns
    prepared_df = working_df[prepared_columns].copy()

    present_legacy_columns = [
        column for column in LEGACY_AUDIT_OPTIONAL if column in working_df.columns
    ]
    legacy_audit_df = working_df[
        METADATA_COLUMNS + present_legacy_columns
    ].copy()

    quality_audit["legacy_columns_present"] = "|".join(present_legacy_columns)
    quality_audit["legacy_columns_missing"] = "|".join(
        sorted(set(LEGACY_AUDIT_OPTIONAL) - set(present_legacy_columns))
    )

    return (
        prepared_df,
        legacy_audit_df,
        schema_audit,
        chronology_audit,
        quality_audit,
    )


## 5. Process all raw stock files

In [6]:

schema_rows = []
chronology_rows = []
quality_rows = []
error_rows = []

run_started = time.perf_counter()

for file_number, file_path in enumerate(raw_files, start=1):
    symbol = normalize_symbol(file_path.stem)

    try:
        (
            prepared_df,
            legacy_audit_df,
            schema_audit,
            chronology_audit,
            quality_audit,
        ) = prepare_one_file(file_path)

        prepared_df.to_csv(
            PREPARED_DATA_PATH / f"{symbol}_prepared.csv",
            index=False,
            encoding=OUTPUT_ENCODING,
        )

        legacy_audit_df.to_csv(
            LEGACY_AUDIT_PATH / f"{symbol}_legacy_audit.csv",
            index=False,
            encoding=OUTPUT_ENCODING,
        )

        schema_rows.append({"symbol": symbol, **schema_audit})
        chronology_rows.append(chronology_audit)
        quality_rows.append(quality_audit)

    except Exception as exc:
        error_rows.append(
            {
                "symbol": symbol,
                "file_name": file_path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if file_number == 1 or file_number % 25 == 0 or file_number == len(raw_files):
        elapsed = time.perf_counter() - run_started
        print(
            f"[{file_number:>4}/{len(raw_files)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(error_rows)}"
        )

elapsed_seconds = time.perf_counter() - run_started
print(f"Completed in {elapsed_seconds:,.1f} seconds.")


[   1/825] elapsed=0.4s errors=0
[  25/825] elapsed=4.9s errors=0
[  50/825] elapsed=9.3s errors=0
[  75/825] elapsed=14.4s errors=0
[ 100/825] elapsed=18.9s errors=0
[ 125/825] elapsed=24.5s errors=0
[ 150/825] elapsed=30.3s errors=0
[ 175/825] elapsed=36.8s errors=0
[ 200/825] elapsed=43.0s errors=0
[ 225/825] elapsed=48.9s errors=0
[ 250/825] elapsed=55.6s errors=0
[ 275/825] elapsed=60.3s errors=0
[ 300/825] elapsed=66.1s errors=0
[ 325/825] elapsed=72.4s errors=0
[ 350/825] elapsed=77.6s errors=0
[ 375/825] elapsed=83.8s errors=0
[ 400/825] elapsed=89.7s errors=0
[ 425/825] elapsed=94.9s errors=0
[ 450/825] elapsed=100.5s errors=0
[ 475/825] elapsed=105.8s errors=0
[ 500/825] elapsed=110.6s errors=0
[ 525/825] elapsed=116.7s errors=0
[ 550/825] elapsed=122.5s errors=0
[ 575/825] elapsed=128.9s errors=0
[ 600/825] elapsed=133.9s errors=0
[ 625/825] elapsed=139.2s errors=0
[ 650/825] elapsed=143.6s errors=0
[ 675/825] elapsed=149.6s errors=0
[ 700/825] elapsed=154.9s errors=0
[ 725/

## 6. Save audit tables

In [7]:

schema_audit_df = pd.DataFrame(schema_rows)
chronology_audit_df = pd.DataFrame(chronology_rows)
row_quality_audit_df = pd.DataFrame(quality_rows)
preparation_errors_df = pd.DataFrame(error_rows)

audit_tables = {
    "01_schema_audit.csv": schema_audit_df,
    "01_chronology_audit.csv": chronology_audit_df,
    "01_row_quality_audit.csv": row_quality_audit_df,
    "01_preparation_errors.csv": preparation_errors_df,
}

for file_name, audit_df in audit_tables.items():
    output_path = RESULT_PATHS["audits"] / file_name
    audit_df.to_csv(output_path, index=False, encoding=OUTPUT_ENCODING)
    print(f"{file_name}: {len(audit_df):,} rows")


01_schema_audit.csv: 825 rows
01_chronology_audit.csv: 825 rows
01_row_quality_audit.csv: 825 rows
01_preparation_errors.csv: 0 rows


## 7. Reproducibility manifest

In [8]:

configuration_snapshot = {
    "paths": paths_config,
    "columns": columns_config,
    "calculate_input_hashes": CALCULATE_INPUT_HASHES,
    "output_encoding": OUTPUT_ENCODING,
}

run_manifest = {
    "notebook": "01_data_preparation.ipynb",
    "stage": "data_preparation",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "input_manifest_hash": input_manifest_hash,
    "configuration_hash": stable_object_hash(configuration_snapshot),
    "raw_files_found": len(raw_files),
    "files_prepared_successfully": len(schema_audit_df),
    "files_failed": len(preparation_errors_df),
    "prepared_output_directory": str(PREPARED_DATA_PATH),
    "legacy_audit_output_directory": str(LEGACY_AUDIT_PATH),
    "elapsed_seconds": elapsed_seconds,
    "software": software_manifest(),
}

run_manifest_path = (
    RESULT_PATHS["manifests"] / "01_data_preparation_run_manifest.json"
)
run_manifest_path.write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("Run manifest:", run_manifest_path)
print(json.dumps(run_manifest, ensure_ascii=False, indent=2, default=str))


Run manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\01_data_preparation_run_manifest.json
{
  "notebook": "01_data_preparation.ipynb",
  "stage": "data_preparation",
  "git_commit_sha": "7aee80ffd9bc4c954311db96494ff31dd78193ce",
  "input_manifest_hash": "1325e202a3686becd739694c4c49902955978b33aeff9357fd436894486c7197",
  "configuration_hash": "811336110d3e1ab32de897dccdfc6a57c5175e70108a34bfe6dd26ec66dfa218",
  "raw_files_found": 825,
  "files_prepared_successfully": 825,
  "files_failed": 0,
  "prepared_output_directory": "E:\\Iran_stock_trade\\financial-ml-decision-support\\data_ready\\prepared",
  "legacy_audit_output_directory": "E:\\Iran_stock_trade\\financial-ml-decision-support\\data_ready\\legacy_audit",
  "elapsed_seconds": 178.78162909997627,
  "software": {
    "timestamp_utc": "2026-07-13T19:49:16.827565+00:00",
    "python_version": "3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)]",
    "platform": "Windows-1

## 8. Final integrity checks

In [9]:

prepared_files = sorted(PREPARED_DATA_PATH.glob("*_prepared.csv"))
legacy_audit_files = sorted(LEGACY_AUDIT_PATH.glob("*_legacy_audit.csv"))

assert preparation_errors_df.empty, (
    "One or more files failed preparation. "
    "Review results/audits/01_preparation_errors.csv."
)

assert len(prepared_files) == len(raw_files)
assert len(legacy_audit_files) == len(raw_files)
assert chronology_audit_df["chronological_order_valid"].all()
assert chronology_audit_df["unique_dates_valid"].all()

print("Notebook 01 integrity checks: PASSED")
print("Prepared files:", len(prepared_files))
print("Legacy audit files:", len(legacy_audit_files))
print("Next stage: Notebook 02 — Temporal split and universe freeze.")


Notebook 01 integrity checks: PASSED
Prepared files: 825
Legacy audit files: 825
Next stage: Notebook 02 — Temporal split and universe freeze.



## Recommended Git checkpoint

After successful execution and review:

```bash
git add .
git commit -m "feat: implement auditable stock data preparation pipeline"
git tag -a v0.1.0-data-preparation -m "Freeze Notebook 01 data preparation"
git push origin main
git push origin v0.1.0-data-preparation
```

Do not create the tag while preparation errors remain unresolved.
